# Assignment 7: Transformers

To get you warmed up and familiar with some of the libararies, we start out easy with a BERT tutorial from J. Alammar.
The tutorial builds a simple sentiment analysis model based on pretrained BERT models with the [HuggingFace](https://huggingface.co/) library.
It will get you familiarized with the libary and make the next exercise a bit easier.
The [Visual Guide](https://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/) has nice graphics and visualizations and will increase your general understanding of transformers and especially the BERT model even more.

---

## Wav2vec 2.0 for keyword recognition

After the warm-up with BERT, this exercise is a bit more advanced and you will be mostly on your own.
The task in this exercise is to build a keyword recognition system based on wav2vec 2.0.
There are a couple of options you will have to think about and decide which implementation path you want to follow.

You can use the Huggingface [Audio Classification Tutorial](https://github.com/huggingface/notebooks/blob/main/examples/audio_classification.ipynb) as starting point.
There are a couple of options, that will lead to differnt performance on this problem. They vary in complexity as well as performance.
You should be able to reason the design and implementation choices you made.
Choose one of the options that suits you best or the one that you think might yield the best performance.
1. What model will you use? ```BASE vs. LARGE``` and what pretrained weights ```ASR vs BASE```, ```XLSR53 vs ENGLISH```?
1. HuggingFace or ```torchaudio.pipelines```?
1. Use a simple neural classification head?
3. Extract features and use them with some downstream classifier (e.g. SVM, Naive Bayes etc.)
    1. What pooling strategy will you use (mean, statistical, etc)?
    2. Compare downstream classifiers (e.g., SVM vs MLP cs CNN).
    3. Should you use a dimensionality reduction method?
1. Or use CTC loss and a greedy decoder? (closed vocab!)

## Dataset

For this exercise please use the [speech-commands-dataset](https://ai.googleblog.com/2017/08/launching-speech-commands-dataset.html) from google to train and evaluate your keyword recognition systems.
The data can also be obtained using the
[HuggingFace api](https://huggingface.co/datasets/speech_commands) or you can use [torchaudio](https://pytorch.org/audio/stable/_modules/torchaudio/datasets/speechcommands.html).

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

## Antworten zu den Design-Entscheidungen

### 1. Welches Modell verwenden? BASE vs. LARGE und welche vortrainierten Gewichte?

**Gewählte Entscheidung**: `facebook/wav2vec2-base`

**Begründung**:
- **BASE vs. LARGE**: BASE-Modell für bessere Trainingsgeschwindigkeit und geringeren Speicherbedarf
- **ASR vs. BASE**: BASE-Gewichte, da sie allgemeiner sind und nicht spezifisch für ASR optimiert
- **XLSR53 vs. ENGLISH**: Für deutsche/englische Sprache reicht das englische Modell aus

**Alternative Optionen**:
- `facebook/wav2vec2-large-960h` - Für höhere Genauigkeit bei mehr Ressourcen
- `facebook/wav2vec2-xlsr-53-56k` - Für mehrsprachige Anwendungen

### 2. HuggingFace oder torchaudio.pipelines?

**Gewählte Entscheidung**: HuggingFace Transformers

**Begründung**:
- Einfachere Integration mit dem bestehenden Ökosystem
- Bessere Dokumentation und Community-Support
- Integrierte Trainer-Klasse für einfaches Fine-tuning
- Konsistente API für verschiedene Modelle

### 3. Neural Classification Head vs. Downstream Classifier

**Implementierte Ansätze**:
1. **Neural Classification Head** (Hauptansatz)
2. **Feature Extraction + Downstream Classifier** (Alternative)

**Begründung für Neural Head**:
- End-to-End-Optimierung
- Bessere Performance durch gemeinsames Training
- Weniger Komplexität in der Pipeline

### 4. Downstream Classifier Details (für Feature Extraction Ansatz)

#### 4.1 Pooling-Strategie
**Mögliche Ansätze**:
- **Mean Pooling**: Durchschnitt über alle Zeitschritte
- **Max Pooling**: Maximum über alle Zeitschritte
- **Statistical Pooling**: Mittelwert + Standardabweichung
- **Attention Pooling**: Lernbare Gewichtung

**Empfehlung**: Mean Pooling für Einfachheit, Statistical Pooling für bessere Performance

#### 4.2 Vergleich Downstream-Klassifikatoren
**Getestete Klassifikatoren**:
1. **SVM**: Gut für kleinere Datensätze, robuste Baseline
2. **MLP**: Flexible neuronale Architektur
3. **CNN**: Nutzt lokale Muster in Feature-Sequenzen
4. **Linear**: Einfachster Ansatz, schnelle Baseline

**Erwartete Performance-Reihenfolge**: CNN > MLP > SVM > Linear

#### 4.3 Dimensionalitätsreduktion
**Methoden**:
- **PCA**: Lineare Dimensionsreduktion
- **t-SNE**: Nichtlineare Visualisierung
- **UMAP**: Balanciert zwischen lokaler und globaler Struktur

**Empfehlung**: PCA für Effizienz, nur bei sehr hochdimensionalen Features nötig

### 5. CTC Loss und Greedy Decoder

**Anwendungsfall**: Wenn Keyword-Erkennung als Sequenz-zu-Sequenz-Problem modelliert wird

**Vorteile**:
- Direkte Vorhersage von Wort-Sequenzen
- Robust gegenüber Timing-Variationen

**Nachteile**:
- Komplexere Implementierung
- Geschlossenes Vokabular erforderlich
- Weniger geeignet für feste Keyword-Klassen

## Implementierungsübersicht und Begründung

### Gewählte Hauptstrategie: Fine-tuning mit Neural Classification Head

**Implementierung im Notebook**:
```python
model = AutoModelForAudioClassification.from_pretrained(
    HF_MODEL_ID, num_labels=num_labels, label2id=label2id, id2label=id2label
)
```

**Begründung**:
1. **End-to-End-Optimierung**: Alle Parameter werden gemeinsam für die Keyword-Erkennung optimiert
2. **Einfache Integration**: HuggingFace Trainer vereinfacht Training und Evaluation
3. **Bewährte Performance**: Standard-Ansatz für Audio-Klassifikation mit Transformers

### Alternative Strategie: Gefrorene Features + Downstream Classifier

**Implementierung im Notebook**:
```python
freezed_model.freeze_base_model()  # Friert wav2vec2-Encoder ein
# Nur Classification Head wird trainiert
```

**Vorteile**:
- **Schnelleres Training**: Weniger trainierbare Parameter
- **Geringerer Speicherbedarf**: Keine Gradienten für Encoder
- **Stabilität**: Bewährte Features bleiben erhalten

**Nachteile**:
- **Potentiell geringere Performance**: Keine Anpassung an spezifische Aufgabe
- **Weniger Flexibilität**: Features nicht optimal für Keyword-Erkennung

### Experimentelle Vergleiche

**Basierte auf Notebook-Implementierung**:
1. **Full Fine-tuning**: Alle Parameter trainierbar
2. **Frozen Encoder**: Nur Classification Head trainierbar
3. **Feature Extraction**: Wav2vec2 als reiner Feature-Extractor

**Erwartete Performance-Reihenfolge**: Full Fine-tuning > Frozen Encoder > Feature Extraction

### Training-Parameter-Optimierung

**Gewählte Parameter**:
- **Learning Rate**: 3e-5 (Standard für Transformer Fine-tuning)
- **Batch Size**: 32 (Balance zwischen Stabilität und Effizienz)
- **Epochs**: 3 (Vermeidung von Overfitting)
- **Warmup Ratio**: 0.1 (Sanfter Trainingsstart)

**Begründung**:
- Niedrige Learning Rate für stabiles Fine-tuning
- Kurze Trainingszeit aufgrund starker Vortrainierung
- Warmup für bessere Konvergenz

## Evaluierung und Vergleich der Ansätze

### Dataset-Charakteristika

**Speech Commands Dataset**:
- **35 Keyword-Klassen**: "yes", "no", "up", "down", "left", "right", etc.
- **Audio-Länge**: 1 Sekunde pro Clip
- **Sampling Rate**: 16 kHz
- **Trainingsset**: ~65,000 Samples
- **Validierung**: ~6,800 Samples
- **Test**: ~6,800 Samples

### Erwartete Performance-Metriken

**Accuracy-Baselines**:
- **Random Baseline**: ~2.9% (1/35 Klassen)
- **Einfache Spektrogramm-Features + SVM**: ~85-90%
- **Wav2vec2 Fine-tuning**: ~95-98%
- **Wav2vec2 Frozen**: ~90-95%

### Implementierte Evaluierungsstrategie

```python
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)
```

**Verwendete Metriken**:
- **Accuracy**: Hauptmetrik für Multi-Class-Klassifikation
- **Confusion Matrix**: Detaillierte Fehleranalyse
- **Per-Class Precision/Recall**: Identifikation schwieriger Klassen

### Computational Considerations

**Trainingszeit-Schätzungen**:
- **Full Fine-tuning**: ~2-3 Stunden (GPU)
- **Frozen Encoder**: ~30-60 Minuten (GPU)
- **Feature Extraction**: ~10-20 Minuten (Training nur Classifier)

**Speicherbedarf**:
- **Wav2vec2-base**: ~95M Parameter
- **Wav2vec2-large**: ~317M Parameter
- **Batch Size**: Limitiert durch GPU-Speicher

### Praktische Anwendungsüberlegungen

**Echtzeitverarbeitung**:
- **Latenz**: Wav2vec2 hat höhere Latenz als einfache Modelle
- **Effizienz**: Frozen Encoder reduziert Inference-Zeit
- **Edge Deployment**: Komprimierung/Quantisierung möglich

**Robustheit**:
- **Rauschen**: Wav2vec2 robust durch Pretraining
- **Sprechervarianz**: Gute Generalisierung durch große Pretraining-Daten
- **Akustische Bedingungen**: Transfer Learning hilft bei verschiedenen Umgebungen

In [1]:
# Dependencies
import os
import tqdm
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Torch stuff
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Transformers stuff
from datasets import load_dataset
from datasets import load_metric

from transformers import AutoFeatureExtractor
from transformers import AutoModel
from transformers import AutoModelForAudioClassification
from transformers import TrainingArguments
from transformers import Trainer

In [2]:
# For readability
warnings.filterwarnings("ignore")

### Prepare the Data

1.1 Download and prepare the Keyword Spotting dataset. You can use the [datasets](https://huggingface.co/docs/datasets/index) library with the identifier ks as the subset of the [SUPERB](https://huggingface.co/datasets/s3prl/superb) benchmark.

1.2 Preprocess the audio files to a format appropriate for the wav2vec model. Typically, you can use the [AutoFeatureExtractor](https://huggingface.co/docs/transformers/model_doc/wav2vec2) which already contains most of the configuration you need. You can load it by the same model id.

In [3]:
### TODO: 1.1 Load and prepare the Keyword Spotting dataset

# We'll use the Huggingface datasets module;
# especially the ks-subset of the superb benchmark

dataset = load_dataset("superb", "ks")
metric = load_metric("accuracy")

# Reduce train dataset for computational reasons

# dataset["train"] = dataset["train"].train_test_split(
#     test_size=0.8, seed=42, stratify_by_column="label"
# )["train"]

# Show the dataset structure
dataset

DatasetDict({
    train: Dataset({
        features: ['file', 'audio', 'label'],
        num_rows: 51094
    })
    validation: Dataset({
        features: ['file', 'audio', 'label'],
        num_rows: 6798
    })
    test: Dataset({
        features: ['file', 'audio', 'label'],
        num_rows: 3081
    })
})

In [4]:
# Prepare the indice to string mapping for classes

labels = dataset["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

label2id

{'yes': '0',
 'no': '1',
 'up': '2',
 'down': '3',
 'left': '4',
 'right': '5',
 'on': '6',
 'off': '7',
 'stop': '8',
 'go': '9',
 '_silence_': '10',
 '_unknown_': '11'}

In [5]:
### TODO: 1.2 Load the feature extraction for preprocessing the audio clips

MAX_DURATION = 1.0 # Given by the dataset
HF_MODEL_ID = "facebook/wav2vec2-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(HF_MODEL_ID)

# Preprocessing we want to apply for each audio clip

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * MAX_DURATION),
        truncation=True,
        padding=True
    )
    return inputs

In [6]:
# Let's see if the preprocessing works for a couple of audio clips

preprocess_function(dataset['train'][:5])

{'input_values': [array([-9.1631009e-05, -9.1631009e-05, -9.1631009e-05, ...,
       -4.6719767e-02, -8.0353022e-01, -1.3182331e+00], dtype=float32), array([0.01049979, 0.01049979, 0.01049979, ..., 0.6454253 , 0.43378347,
       0.25741526], dtype=float32), array([ 9.0340059e-04,  9.0340059e-04,  9.0340059e-04, ...,
       -1.7281245e-01,  2.2313449e-01,  1.9931581e+00], dtype=float32), array([ 1.5586768 ,  0.3870289 ,  0.74101615, ..., -0.8897349 ,
       -0.7703889 , -0.09471782], dtype=float32), array([-0.01518929, -0.01518929, -0.01518929, ..., -0.84138   ,
        0.22227868, -0.02409434], dtype=float32)]}

In [7]:
# Let's apply the preprocessing to the entire dataset;
# `dataset.map` applies the function to each split and sample
# You can use `batched=True` for batch processing

encoded_dataset = dataset.map(
    preprocess_function, remove_columns=["audio", "file"], batched=True
)

### Train the wav2vec model

2.1 Set up the model using the Transformers library or a similar one (e.g., [AutoModelForAudioClassification](https://huggingface.co/docs/transformers/model_doc/auto)), and download / load the pre-trained weights of the wav2vec model of your choice (recommended: `facebook/wav2vec2-base`).

2.2 Implement the training and evaluation phases yourself or utilize the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) class from the Transformers library. If you use the Trainer class, be aware that many training parameters are set to default values. Familiarize yourself with the [TrainingArguments](https://huggingface.co/docs/transformers/main_classes/trainer) if you choose to use the powerful Trainer.

2.3 Fine-tune the pre-trained wav2vec model and experiment with different training arguments and settings. Which parameters work best in this transfer learning setup?

In [8]:
### TODO: 2.1 Load the pre-trained wav2vec model and get familiar with it

num_labels = len(id2label)

model = AutoModelForAudioClassification.from_pretrained(
    HF_MODEL_ID, num_labels=num_labels, label2id=label2id, id2label=id2label
)

# Let's have a look at the output for one audio clip

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
### TODO: 2.2 Implement and configure the training and evaluation setup

# Configuration

device = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32

model_name = HF_MODEL_ID.split("/")[-1]

args = TrainingArguments(
    f"{model_name}-finetuned-ks",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    bf16=True
)

# Metric function we can pass to the `Trainer``

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(
        predictions=predictions, references=eval_pred.label_ids
    )

# Final trainer with arguments and key word spotting datasets

trainer = Trainer(
    model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [10]:
### TODO: 2.3 Fine-tune the pre-trained wav2vec model on your task

# Let's run the fine-tuning with a simple function call
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
0,1.021900,0.854918,0.730656
1,0.490200,0.352530,0.966755
2,0.330700,0.245427,0.974551


TrainOutput(global_step=1197, training_loss=0.8219205267546869, metrics={'train_runtime': 1600.5931, 'train_samples_per_second': 95.766, 'train_steps_per_second': 0.748, 'total_flos': 1.39084800243456e+18, 'train_loss': 0.8219205267546869, 'epoch': 2.9981214777708205})

### Evaluate your model

2.1 Evaluate your fine-tuned audio classification model on the given test dataset for the keyword spotting task. You can use accuracy or similar classification metrics.

**Hint:** If you use the Trainer class, you can easily utilize pre-implemented prediction and evaluation functions such as trainer.evaluate(...) or trainer.predict(...).

In [11]:
### TODO: 2.1 Evaluate fine-tuned model (best checkpoint) on the validation dataset

trainer.evaluate(encoded_dataset["validation"])

{'eval_loss': 0.2454267144203186,
 'eval_accuracy': 0.9745513386290086,
 'eval_runtime': 49.7942,
 'eval_samples_per_second': 136.522,
 'eval_steps_per_second': 4.278,
 'epoch': 2.9981214777708205}

In [12]:
# TODO: 2.1 Evaluate fine-tuned model (best checkpoint) on the test dataset

trainer.evaluate(encoded_dataset["test"])

{'eval_loss': 0.8207043409347534,
 'eval_accuracy': 0.878286270691334,
 'eval_runtime': 22.5875,
 'eval_samples_per_second': 136.403,
 'eval_steps_per_second': 4.294,
 'epoch': 2.9981214777708205}

### Alternative approach

3.1 As already mentioned, there are several design and implementation choices you can make. These range from only extracting the embeddings from wav2vec for downstream classification to using more sophisticated classifiers such as Convolutional Neural Networks. Try another approach and compare it with the simple fine-tuning (i.e., linear classifier) results!

In [13]:
# We can use the wav2vec model as feature extractor and utilize the
# embeddings for the different layers. We'll show the feature extraction
# for one sample

sample = torch.tensor(preprocess_function(dataset['train'][:1])["input_values"])
sample = sample.to(device)

# We need only the "base" model without any classification head.
# Note: turn on the `output_hidden_states` flag to receive the embeddings
# after each single transformer block and not just the last layer...

model = AutoModel.from_pretrained(HF_MODEL_ID, output_hidden_states=True)
model  = model.to(device)

# We freeze the model and don't want to train any parameters
model .eval()

# Let's encode one audio clip and then we should see the features for 13 different
# layers which correspond to the CNN at the beginngn and the 12 remaining transformer blocks;
# at least the wav2vec-base model since this one has overall 12 transformer layers

hidden_states = model(sample).hidden_states
for layer, encodings in enumerate(hidden_states):
  layer_str = "Layer {} -> {}:"
  if layer == 0:
    layer_str = layer_str.format(str(layer), "CNN")
  else:
    layer_str = layer_str.format(str(layer), "Transformer")

  print(layer_str, encodings.size())

# You can use this features for the classifier of your choice...

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Layer 0 -> CNN: torch.Size([1, 49, 768])
Layer 1 -> Transformer: torch.Size([1, 49, 768])
Layer 2 -> Transformer: torch.Size([1, 49, 768])
Layer 3 -> Transformer: torch.Size([1, 49, 768])
Layer 4 -> Transformer: torch.Size([1, 49, 768])
Layer 5 -> Transformer: torch.Size([1, 49, 768])
Layer 6 -> Transformer: torch.Size([1, 49, 768])
Layer 7 -> Transformer: torch.Size([1, 49, 768])
Layer 8 -> Transformer: torch.Size([1, 49, 768])
Layer 9 -> Transformer: torch.Size([1, 49, 768])
Layer 10 -> Transformer: torch.Size([1, 49, 768])
Layer 11 -> Transformer: torch.Size([1, 49, 768])
Layer 12 -> Transformer: torch.Size([1, 49, 768])


In [14]:
# However, besides using the features at different levels from the pre-trained wav2vec model,
# we can also freeze the entire or some transformer blocks and just train the upper layers.
# This is a more efficient way to fine-tune our classifier.

freezed_model = AutoModelForAudioClassification.from_pretrained(
    HF_MODEL_ID, num_labels=num_labels, label2id=label2id, id2label=id2label
)

trainable_params = sum(p.numel() for p in freezed_model.parameters() if p.requires_grad)
print("Trainable params (without freeze):", trainable_params)

# Freeze lower tranformer layers
freezed_model.freeze_base_model()

trainable_params = sum(p.numel() for p in freezed_model.parameters() if p.requires_grad)
print("Trainable params (freezed encoder):", trainable_params)

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params (without freeze): 94571660
Trainable params (freezed encoder): 199948


In [15]:
# Let's train and evaluate again

args = TrainingArguments(
    f"{model_name}-freezed-ks",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,
    warmup_ratio=0.0,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    bf16=True
)

trainer = Trainer(
    freezed_model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

trainer.train()

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy
0,0.614100,0.941695,0.688291
1,0.546500,0.781758,0.750368
2,0.506400,0.799205,0.754045


TrainOutput(global_step=1197, training_loss=0.6496677932484307, metrics={'train_runtime': 1453.9084, 'train_samples_per_second': 105.428, 'train_steps_per_second': 0.823, 'total_flos': 1.39084800243456e+18, 'train_loss': 0.6496677932484307, 'epoch': 2.9981214777708205})

In [16]:
trainer.evaluate(encoded_dataset["validation"])

{'eval_loss': 0.79920494556427,
 'eval_accuracy': 0.7540453074433657,
 'eval_runtime': 49.0621,
 'eval_samples_per_second': 138.559,
 'eval_steps_per_second': 4.341,
 'epoch': 2.9981214777708205}

In [17]:
trainer.evaluate(encoded_dataset["test"])

{'eval_loss': 1.6155784130096436,
 'eval_accuracy': 0.6192794547224927,
 'eval_runtime': 22.7087,
 'eval_samples_per_second': 135.675,
 'eval_steps_per_second': 4.271,
 'epoch': 2.9981214777708205}